# Security Guide Evaluation

We're creating a custom cybersecurity evaluation inspired by Inspect's “Security Guide” evaluation. We're testing how effectively large language models provide accurate, concise cybersecurity guidance using 16 question-answer pairs from the original OpenAI Evals Security Guide dataset. For example, one of the questions listed is: “How should I securely store passwords?”. The expected ideal answer is “strong hashing algorithms with salt like Argon2 or bcrypt”. The original Inspect evaluation can be found at: https://inspect.aisi.org.uk/tutorial.html

You'll also need an Anthropic API token, which you can get by creating an account at console.anthropic.com. You will need a credit card but it’s pretty cheap: running Claude multiple times to check these demos only cost me 2 cents (USD).

We start by installing the required Python packages: anthropic, for interacting with Anthropic's Claude API, and tqdm, which provides progress bars for loops to visually track processing.

In [ ]:
!pip install anthropic tqdm

We now import the essential Python libraries: anthropic to access Anthropic's Claude API; os for accessing environment variables like API keys; pandas and json to structure and process the dataset; tqdm to show visual progress during processing; and requests to retrieve data files remotely via HTTP.

We also set your Anthropic API key - replace “YOUR-API-KEY” with your actual API token - and initializes the Anthropic client, which enables you to interact with Claude via the API.

In [ ]:
import anthropic
import os
import pandas as pd
import json
from tqdm import tqdm
import requests

api_key = "" #Insert here
client = anthropic.Anthropic(api_key=api_key)

In [ ]:
completion = client.messages.create(
    model="claude-3-haiku-20240307",
    messages=[{"role": "user", "content": "Hello, Claude!"}],
    max_tokens=20
)

print(completion.content[0].text)

Hello! How can I assist you today?


We can try other models, too. This code lets you select and use one of Anthropic’s Claude 3 models by commenting/uncommenting the relevant line. The available models include Claude 3 Haiku (the smallest at around tens of billions of parameters), Claude 3 Sonnet (the low hundreds of billions of parameters), and Claude 3 Opus (better for complex reasoning tasks at several hundred billion parameters). The numbers in each model name represent the release date of that specific model version (in YYYYMMDD format).

In [ ]:
# Choose your Claude 3 model by uncommenting one of these lines
#model_name = "claude-3-haiku-20240307"  # Big
#model_name = "claude-3-sonnet-20240229"  # Bigger
model_name = "claude-3-opus-20240229"  # Biggest

Now we retrieve the security evaluation dataset directly from my GitHub. I had already downloaded this from the Inspect GitHub (just in case something happens and it’s not available from the same link). It explicitly checks the HTTP request for errors using raise_for_status(). Each line of the downloaded JSONL file is parsed individually, then converted into a structured pandas DataFrame (df_eval). Finally, it clearly prints out the dataset's size and previews the first few entries, ensuring the data is loaded correctly and ready for evaluation.

In [ ]:
dataset_url = "https://raw.githubusercontent.com/harrietf/book/main/security_samples.jsonl"

response = requests.get(dataset_url)
response.raise_for_status()

data = [json.loads(line.strip()) for line in response.text.strip().split('\n')]
df_eval = pd.DataFrame(data)

print(f"Dataset size: {len(df_eval)} entries")
df_eval.head()

Dataset size: 16 entries


,input,ideal
0,"[{'role': 'system', 'content': 'Please provide...",use parameterized queries and prepared statements
1,"[{'role': 'system', 'content': 'Please provide...",use parameterized queries and prepared statements
2,"[{'role': 'system', 'content': 'Please provide...",output encoding with secure by default librari...
3,"[{'role': 'system', 'content': 'Please provide...",output encoding with secure by default librari...
4,"[{'role': 'system', 'content': 'Please provide...",parameterized commands and input validation


The function compares two inputs (expected and actual) by breaking them into lowercase keywords, finding common words (overlap), and computing the ratio of matched keywords to total expected keywords, resulting in a numerical accuracy score between 0 and 1. Next, the code loops through each entry of our cybersecurity evaluation dataset (df_eval), extracting a system_prompt and a user_prompt (the specific security question). It sends these prompts to the chosen Claude model using Anthropic's API with a token limit (max_tokens=50). The generated response from Claude is retrieved, trimmed, and stored in a list named responses for scoring and analysis.

In [ ]:
# Keyword overlap scoring function
def keyword_overlap(expected, actual):
    expected_keywords = set(expected.lower().split())
    actual_keywords = set(actual.lower().split())
    overlap = expected_keywords & actual_keywords
    return len(overlap) / len(expected_keywords) if expected_keywords else 0

# Generate responses and calculate scores
responses = []

for idx, row in df_eval.iterrows():
    system_prompt = row['input'][0]['content']
    user_prompt = row['input'][1]['content']

    completion = client.messages.create(
        model=model_name,
        system=system_prompt,
        messages=[{"role": "user", "content": user_prompt}],
        max_tokens=50
    )

    response = completion.content[0].text.strip()
    responses.append(response)

/tmp/ipython-input-23-1554867401.py:15: DeprecationWarning: The model 'claude-3-opus-20240229' is deprecated and will reach end-of-life on January 5th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  completion = client.messages.create(


The resulting scores (between 0 and 1, where 1 indicates a perfect keyword match) are saved to another new column, score. Finally, the code neatly displays the first few entries of the updated dataset, clearly showing the original input prompts, ideal answers, Claude’s generated responses, and corresponding accuracy scores for easy comparison and analysis.

In [ ]:
# Store Claude's responses explicitly
df_eval['claude_response'] = responses

# Calculate scores explicitly
df_eval['score'] = df_eval.apply(
    lambda row: keyword_overlap(row['ideal'], row['claude_response']),
    axis=1
)

# View results clearly
df_eval[['input', 'ideal', 'claude_response', 'score']].head()


,input,ideal,claude_response,score
0,"[{'role': 'system', 'content': 'Please provide...",use parameterized queries and prepared statements,To prevent SQL injection attacks:\n- Validate ...,0.666667
1,"[{'role': 'system', 'content': 'Please provide...",use parameterized queries and prepared statements,To prevent SQL injection (SQLi):\n- Use parame...,0.666667
2,"[{'role': 'system', 'content': 'Please provide...",output encoding with secure by default librari...,To prevent cross-site scripting (XSS):\n• Vali...,0.300000
3,"[{'role': 'system', 'content': 'Please provide...",output encoding with secure by default librari...,To prevent cross-site scripting (XSS):\n- Vali...,0.400000
4,"[{'role': 'system', 'content': 'Please provide...",parameterized commands and input validation,To prevent command injection:\n- Validate and ...,0.600000


This code calculates the average accuracy score for all of Claude’s responses across the cybersecurity evaluation dataset, using the score column created earlier. It then neatly prints this average score to give an overall measure of Claude’s performance. Next, it explicitly loops through each entry in the evaluation dataset (up to 16 examples), clearly printing out the original prompt, the ideal expected answer, Claude’s generated response, and the associated keyword-overlap score, facilitating easy analysis of each response and how closely it matches cybersecurity best practices.

In [ ]:
# Calculate average score explicitly
average_score = df_eval['score'].mean()
print(f"\nAverage Evaluation Score: {average_score:.2f}\n")

# Clearly display examples with the 'ideal' column
for idx, row in df_eval.head(16).iterrows():
    print(f"Prompt {idx+1}: {row['input']}")
    print(f"Expected Answer: {row['ideal']}")  # explicitly updated here
    print(f"Claude's Response: {row['claude_response']}")
    print(f"Score: {row['score']:.2f}")
    print('-' * 50)



Average Evaluation Score: 0.38

Prompt 1: [{'role': 'system', 'content': 'Please provide good cybersecurity guidance. Provide short response in a few words.'}, {'role': 'user', 'content': 'How do I prevent SQL Injection attacks?'}]
Expected Answer: use parameterized queries and prepared statements
Claude's Response: To prevent SQL injection attacks:
- Validate and sanitize user input 
- Use parameterized queries
- Limit database privileges
- Employ web application firewalls
Score: 0.67
--------------------------------------------------
Prompt 2: [{'role': 'system', 'content': 'Please provide good cybersecurity guidance. Provide short response in a few words.'}, {'role': 'user', 'content': 'How do I prevent sqli?'}]
Expected Answer: use parameterized queries and prepared statements
Claude's Response: To prevent SQL injection (SQLi):
- Use parameterized queries 
- Validate and sanitize user input
- Limit database privileges
- Use stored procedures
- Implement web application firewalls
S